# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates step-by-step exploration, extraction, processing, and basic visualization of the FAIR^2 dataset, guided by the [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and using the [`mlcroissant`](https://mlcroissant.org) Python library.

### Dataset Source
The dataset's schema and data description are provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`. This step will fetch the structured data and its schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL of the Croissant schema for the FAIR^2 dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset high-level metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', '-')}")
print(f"Published: {getattr(metadata, 'datePublished', '-')}")
print(f"License: {getattr(metadata, 'license', '-')}")

## 2. Data Overview
Explore available record sets and their fields (columns). Every entity is referenced by its `@id` as required by the Croissant specification.

Below, we print all available record sets (by `@id` and name), their available fields (also by `@id`), and the corresponding column ids (if any) for each field.

In [ ]:
# List all record sets, their fields (@id), and columns (@id)
print("Record Sets Overview:\n--------------------")
record_sets = []
for rs in metadata.record_sets:
    print(f"@id: {rs.id}")
    print(f"  Name: {rs.name}")
    if hasattr(rs, 'description'):
        print(f"  Description: {rs.description}")

    # List fields by @id
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id} ; name: {field.name}; type: {getattr(field, 'data_type', getattr(field, 'dataType', '-'))}")

        # List associated columns by @id (if present)
        if hasattr(field, 'columns') and field.columns:
            for col in field.columns:
                print(f"        column @id: {col.id} ; name: {col.name}")
    print()
    record_sets.append(rs.id)
print(f"All record sets: {record_sets}")

## 3. Data Extraction
Load the records (tabular data) from a chosen record set into a pandas DataFrame. Use `@id` references for the record set and fields, and print sample data and column information.

For this dataset, we will extract all available record sets by their `@id` into separate DataFrames.

In [ ]:
# Extract records (tables) for all record sets (referenced by @id)
dfs = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dfs[record_set_id] = df
    print(f"Loaded record set @id: {record_set_id} with shape {df.shape}")

# Show columns (fields) for each record set
for recset_id, df in dfs.items():
    print(f"\nColumns for record set {recset_id}:\n{df.columns.tolist()}")
    display(df.head())  # You can comment out display for script mode

## 4. Exploratory Data Analysis (EDA)
Perform some typical data processing operations. We will:
- Filter a numeric field (e.g., age or diagnostic interval) for outliers,
- Normalize that field,
- And group by a relevant categorical field.

**NOTE:** In this section, you must look at the previous code output to decide which record set/field `@id`s to use for numeric and grouping fields.

For demonstration, we assume a field called `age_at_second_crc_diagnosis` and grouping by `sex`. Please adjust field ids based on EDA above if different.

In [ ]:
# Choose the main patient-level record set (@id)
# For this dataset, we'll assume only one main table -- adjust below based on the output of the previous step.
main_record_set_id = record_sets[0]
df = dfs[main_record_set_id]

# Identify numeric and group fields by their @id or dataframe column name (from the Data Overview step)
# Example names, you may need to replace these with real field/column @id values
numeric_field_id = 'age_at_second_crc_diagnosis'  # Replace with actual @id or column name if different
group_field_id = 'sex'                            # Replace with actual @id or column name if different

if numeric_field_id in df.columns:
    # Try to ensure numeric type
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Filter to records where age > 40
    threshold = 40
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize numeric field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}':")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
        display(grouped_df)
    else:
        print(f"Group field '{group_field_id}' not found in this record set.")
else:
    print(f"Numeric field '{numeric_field_id}' not found in this record set. Please use a correct @id from your dataset.")

## 5. Visualization
Visualize data distributions and relationships between fields. Example: plot the distribution of the numeric field and compare by group, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

# Boxplot by group
if (numeric_field_id in df.columns) and (group_field_id in df.columns):
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we have demonstrated how to use the [mlcroissant](https://mlcroissant.org) Python library to:
- Load and inspect metadata and record sets directly from a Croissant schema (`@id` referenced throughout),
- Extract data into pandas DataFrames for each record set,
- Perform exploratory data analysis (EDA) including filtering, normalization, grouping,
- Visualize main distributions and group relationships.

This approach ensures reproducibility and proper referencing of all relevant data and schema items in alignment with FAIR principles, as implemented in the FAIR^2 dataset.

For detailed modeling and analysis, further data exploration, statistical tests, or ML could follow using these prepared DataFrames and the fully specified Croissant structure.